# Detección de Drift en Sistemas LLM

> Aprende a monitorizar la degradación del rendimiento de LLMs en producción
> detectando drift de datos, drift de concepto y degradación de calidad.

## Objetivos
- Detectar drift en la distribución de prompts de entrada
- Monitorizar métricas de calidad a lo largo del tiempo
- Implementar alertas automáticas cuando el rendimiento cae
- Visualizar dashboards de monitorización en producción

## 1. Instalación de dependencias

In [ ]:
%pip install anthropic scipy pandas matplotlib numpy --quiet

## 2. Tipos de drift en sistemas LLM

In [ ]:
import anthropic
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
from scipy import stats
from datetime import datetime, timedelta
from collections import defaultdict

client = anthropic.Anthropic()
MODELO = "claude-haiku-4-5-20251001"

print("=== TIPOS DE DRIFT EN SISTEMAS LLM ===")
print("""
1. DATA DRIFT (distribución de entradas):
   - Cambia el tipo de preguntas que hacen los usuarios
   - Cambia el idioma, longitud o vocabulario de los prompts
   - Detección: comparar estadísticas de prompts baseline vs. recientes

2. CONCEPT DRIFT (el mundo cambia):
   - La información correcta cambia (precios, leyes, productos)
   - El modelo da respuestas desactualizadas
   - Detección: evaluación periódica con dataset de referencia actualizado

3. QUALITY DRIFT (degradación de calidad):
   - Las respuestas se vuelven menos precisas o más largas
   - Cambios en el modelo base (actualizaciones del proveedor)
   - Detección: monitorizar métricas de calidad en ventanas temporales

4. FEEDBACK DRIFT:
   - Los usuarios dan menos feedback positivo
   - Aumentan las quejas o re-preguntas
   - Detección: monitorizar señales implícitas de satisfacción
""")

## 3. Simular datos de producción con drift

In [ ]:
np.random.seed(42)

# Simular 30 días de producción con drift en la semana 3
dias = 30
llamadas_por_dia = 20

# Baseline: preguntas cortas sobre soporte técnico
temas_baseline = ["contraseña", "factura", "envío", "devolución", "cuenta"]
# Después del drift: preguntas largas y complejas (nueva campaña de marketing)
temas_drift = ["configuración avanzada", "integración API", "personalización", "automatización", "reportes"]

registros = []
fecha_base = datetime(2024, 1, 1)

for dia in range(dias):
    fecha = fecha_base + timedelta(days=dia)
    # Drift a partir del día 20
    hay_drift = dia >= 20
    temas = temas_drift if hay_drift else temas_baseline

    for _ in range(llamadas_por_dia):
        tema = np.random.choice(temas)
        longitud_prompt = np.random.normal(150 if hay_drift else 60, 20)
        latencia = np.random.normal(0.8 if hay_drift else 0.5, 0.1)
        # Calidad cae después del drift (modelo no preparado para preguntas complejas)
        calidad = np.random.normal(6.5 if hay_drift else 8.2, 1.0)
        tokens_salida = np.random.normal(180 if hay_drift else 80, 30)

        registros.append({
            "fecha": fecha.date(),
            "dia": dia,
            "tema": tema,
            "longitud_prompt": max(20, longitud_prompt),
            "latencia_s": max(0.1, latencia),
            "calidad_juez": min(10, max(1, calidad)),
            "tokens_salida": max(20, tokens_salida),
            "con_drift": hay_drift,
        })

df = pd.DataFrame(registros)
print(f"Dataset generado: {len(df)} registros, {dias} días")
print(f"Período baseline: días 0-19 ({llamadas_por_dia*20} llamadas)")
print(f"Período con drift: días 20-29 ({llamadas_por_dia*10} llamadas)")
print(f"\nEstadísticas globales:")
print(df.groupby("con_drift")[["longitud_prompt", "calidad_juez", "tokens_salida"]].mean().round(1))

## 4. Detectar drift con tests estadísticos

In [ ]:
def detectar_drift_ks(baseline: np.ndarray, reciente: np.ndarray,
                      metrica: str, umbral_p: float = 0.05) -> dict:
    """Kolmogorov-Smirnov test para detectar drift en una métrica."""
    stat, p_valor = stats.ks_2samp(baseline, reciente)
    drift_detectado = p_valor < umbral_p
    return {
        "metrica": metrica,
        "estadistico_ks": round(stat, 4),
        "p_valor": round(p_valor, 4),
        "drift_detectado": drift_detectado,
        "media_baseline": round(baseline.mean(), 3),
        "media_reciente": round(reciente.mean(), 3),
        "cambio_relativo": f"{(reciente.mean() - baseline.mean()) / baseline.mean():.1%}"
    }

baseline = df[df["dia"] < 20]
reciente = df[df["dia"] >= 20]

metricas_drift = [
    ("longitud_prompt", "Data drift (longitud prompt)"),
    ("calidad_juez", "Quality drift (calidad)"),
    ("tokens_salida", "Output drift (tokens generados)"),
    ("latencia_s", "Performance drift (latencia)"),
]

print("=== DETECCIÓN DE DRIFT (Kolmogorov-Smirnov) ===")
resultados_drift = []
for col, descripcion in metricas_drift:
    resultado = detectar_drift_ks(
        baseline[col].values, reciente[col].values, descripcion
    )
    resultados_drift.append(resultado)
    icono = "🔴" if resultado["drift_detectado"] else "🟢"
    print(f"\n{icono} {descripcion}")
    print(f"   Baseline: {resultado['media_baseline']} → Reciente: {resultado['media_reciente']} ({resultado['cambio_relativo']})")
    print(f"   KS={resultado['estadistico_ks']}, p={resultado['p_valor']} → {'DRIFT DETECTADO' if resultado['drift_detectado'] else 'Sin drift significativo'}")

# Ventana deslizante para detección temprana
print("\n=== DETECCIÓN EN VENTANA DESLIZANTE (7 días) ===")
df_diario = df.groupby("dia")["calidad_juez"].agg(["mean", "std"]).reset_index()
df_diario["alerta"] = df_diario["mean"] < 7.5  # umbral de calidad
alertas = df_diario[df_diario["alerta"]]
if len(alertas) > 0:
    print(f"⚠ Días con calidad por debajo del umbral (7.5): {list(alertas['dia'].values)}")
else:
    print("✓ Sin alertas de calidad")

## 5. Dashboard de monitorización

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Dashboard de Monitorización LLM — 30 días", fontsize=14, fontweight="bold")

df_diario_full = df.groupby("dia").agg(
    calidad_media=("calidad_juez", "mean"),
    latencia_media=("latencia_s", "mean"),
    longitud_media=("longitud_prompt", "mean"),
    tokens_medio=("tokens_salida", "mean"),
).reset_index()

dias_x = df_diario_full["dia"].values

for ax, metrica, titulo, umbral, color in [
    (axes[0, 0], "calidad_media", "Calidad (LLM-judge)", 7.5, "#3498db"),
    (axes[0, 1], "latencia_media", "Latencia media (s)", 1.0, "#e74c3c"),
    (axes[1, 0], "longitud_media", "Longitud promedio del prompt", None, "#2ecc71"),
    (axes[1, 1], "tokens_medio", "Tokens generados (media)", None, "#9b59b6"),
]:
    valores = df_diario_full[metrica].values
    ax.plot(dias_x, valores, color=color, linewidth=2)
    ax.fill_between(dias_x, valores, alpha=0.2, color=color)
    ax.axvline(x=20, color="red", linestyle="--", alpha=0.7, label="Inicio drift")
    if umbral:
        ax.axhline(y=umbral, color="orange", linestyle=":", alpha=0.8, label=f"Umbral ({umbral})")
    ax.set_title(titulo, fontweight="bold")
    ax.set_xlabel("Día")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("drift_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n=== SISTEMA DE ALERTAS ===")
print("""
Reglas de alerta recomendadas:
  🔴 CRÍTICO: calidad < 6.0 durante 3 días consecutivos
  🟠 ALTO: drift KS detectado en longitud de prompts (p < 0.01)
  🟡 MEDIO: latencia > percentil 95 del baseline
  🟢 INFO: cambio > 20% en distribución de temas

Acciones recomendadas al detectar drift:
  1. Analizar muestras recientes del drift
  2. Comparar con Golden Dataset de evaluación
  3. Considerar fine-tuning o actualización del system prompt
  4. Escalar revisión humana temporalmente
""")